# EDA-7 — Signal Synthesis

Orchestration notebook. Assembles governed characterization outputs from EDA-3 through EDA-6
into a compact synthesis artifact at the canonical (signal, position, population_scope) grain.

**Output:**
- `research/eda/findings/eda_07_signal_synthesis.csv`

**Depends on:** EDA-3, EDA-4, EDA-5, EDA-6 findings.

This notebook is an assembly layer only. No analytical statistics are recomputed here.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from core.governance.promotion import (
    assign_promotion_class,
    PROMOTION_CLASS_VALUES,
)

## Paths and constants

All input/output paths and controlled vocabularies are defined here.

In [2]:
FINDINGS_DIR = Path("../findings")

# Input paths
JOINT_REGISTRY_PATH    = FINDINGS_DIR / "eda_03_joint_registry.csv"
POPULATION_VALIDITY_PATH = FINDINGS_DIR / "eda_04_population_validity.csv"
POOLING_DECISIONS_PATH = FINDINGS_DIR / "eda_05_pooling_decisions.csv"
CONSTRUCT_MAP_PATH     = FINDINGS_DIR / "eda_06_construct_map.csv"
PARTIAL_RHO_PATH       = FINDINGS_DIR / "eda_06_partial_rho.csv"

# Output path
SYNTHESIS_OUTPUT_PATH  = FINDINGS_DIR / "eda_07_signal_synthesis.csv"

# Controlled merge keys
SIGNAL_POSITION_KEYS        = ["signal", "position"]
SYNTHESIS_GRAIN_KEYS        = ["signal", "position", "population_scope"]

# Required columns per upstream input
JOINT_REGISTRY_REQUIRED_COLS = {
    "signal", "position", "population_scope",
    "preferred_population", "downstream_status", "association_class", "signal_layer",
}
POPULATION_VALIDITY_REQUIRED_COLS = {"signal", "position", "population_robustness"}
POOLING_DECISIONS_REQUIRED_COLS   = {"signal", "position", "homogeneity"}
CONSTRUCT_MAP_REQUIRED_COLS       = {"signal_a", "signal_b", "position"}
PARTIAL_RHO_REQUIRED_COLS         = {"signal_a", "signal_b", "position"}

# Final synthesis output columns (ordered)
SYNTHESIS_OUTPUT_COLS = [
    "signal",
    "position",
    "population_scope",
    "population_robustness",
    "preferred_population",
    "temporal_stability",
    "association_class",
    "downstream_status",
    "promotion_class",
    "construct_relationship_present",
]

print(f"Findings dir:    {FINDINGS_DIR.resolve()}")
print(f"Output:          {SYNTHESIS_OUTPUT_PATH}")
print(f"Promotion class vocabulary: {sorted(PROMOTION_CLASS_VALUES)}")

Findings dir:    /Users/safarifgisa/Documents/fpl-intelligence/research/eda/findings
Output:          ../findings/eda_07_signal_synthesis.csv
Promotion class vocabulary: ['context_control', 'core_signal', 'exposure_control', 'market_context', 'review_signal']


## 1. Load upstream inputs

Fails early if any required file is missing.

In [3]:
required_inputs = {
    "eda_03_joint_registry":    JOINT_REGISTRY_PATH,
    "eda_04_population_validity": POPULATION_VALIDITY_PATH,
    "eda_05_pooling_decisions":  POOLING_DECISIONS_PATH,
    "eda_06_construct_map":      CONSTRUCT_MAP_PATH,
    "eda_06_partial_rho":        PARTIAL_RHO_PATH,
}

missing_files = [name for name, path in required_inputs.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Required upstream files missing: {missing_files}\n"
        "Run EDA-3 through EDA-6 before executing EDA-7."
    )

joint_registry_df   = pd.read_csv(JOINT_REGISTRY_PATH)
population_validity_df = pd.read_csv(POPULATION_VALIDITY_PATH)
pooling_decisions_df = pd.read_csv(POOLING_DECISIONS_PATH)
construct_map_df    = pd.read_csv(CONSTRUCT_MAP_PATH)
partial_rho_df      = pd.read_csv(PARTIAL_RHO_PATH)

print(f"joint_registry:       {len(joint_registry_df):,} rows")
print(f"population_validity:  {len(population_validity_df):,} rows")
print(f"pooling_decisions:    {len(pooling_decisions_df):,} rows")
print(f"construct_map:        {len(construct_map_df):,} rows")
print(f"partial_rho:          {len(partial_rho_df):,} rows")

joint_registry:       116 rows
population_validity:  116 rows
pooling_decisions:    116 rows
construct_map:        33 rows
partial_rho:          33 rows


## 2. Validate upstream inputs

In [4]:
# Assert required columns exist in each upstream input.
checks = [
    ("joint_registry",      joint_registry_df,      JOINT_REGISTRY_REQUIRED_COLS),
    ("population_validity", population_validity_df,  POPULATION_VALIDITY_REQUIRED_COLS),
    ("pooling_decisions",   pooling_decisions_df,    POOLING_DECISIONS_REQUIRED_COLS),
    ("construct_map",       construct_map_df,        CONSTRUCT_MAP_REQUIRED_COLS),
    ("partial_rho",         partial_rho_df,          PARTIAL_RHO_REQUIRED_COLS),
]
for name, df, required_cols in checks:
    missing_cols = required_cols - set(df.columns)
    assert not missing_cols, (
        f"{name} is missing required columns: {sorted(missing_cols)}"
    )

# Assert no null signal names or positions in the base registry.
assert joint_registry_df["signal"].isna().sum() == 0, "joint_registry has null signal names"
assert joint_registry_df["position"].isna().sum() == 0, "joint_registry has null positions"

# Assert no duplicate synthesis keys in the base registry.
registry_dups = int(joint_registry_df[SYNTHESIS_GRAIN_KEYS].duplicated().sum())
assert registry_dups == 0, (
    f"joint_registry has {registry_dups} duplicate (signal, position, population_scope) rows"
)

# Assert no duplicate (signal, position) keys in EDA-4 and EDA-5.
eda4_dups = int(population_validity_df[SIGNAL_POSITION_KEYS].duplicated().sum())
assert eda4_dups == 0, f"population_validity has {eda4_dups} duplicate (signal, position) rows"

eda5_dups = int(pooling_decisions_df[SIGNAL_POSITION_KEYS].duplicated().sum())
assert eda5_dups == 0, f"pooling_decisions has {eda5_dups} duplicate (signal, position) rows"

print("All upstream input validations passed.")

All upstream input validations passed.


## 3. Build synthesis frame

Canonical grain: (signal, position, population_scope).

All joins use explicit merge keys with cardinality assertions.

In [5]:
# Base frame from EDA-3 joint registry — select only fields needed for synthesis.
# signal_layer is retained temporarily for assign_promotion_class; dropped before output.
base = joint_registry_df[
    ["signal", "position", "population_scope", "preferred_population",
     "downstream_status", "association_class", "signal_layer"]
].copy()

n_base = len(base)
print(f"Base frame rows: {n_base}")

# --- Join EDA-4: population_robustness per (signal, position) ---
eda4 = population_validity_df[["signal", "position", "population_robustness"]].copy()
merged = base.merge(eda4, on=SIGNAL_POSITION_KEYS, how="left")
assert len(merged) == n_base, (
    f"EDA-4 join inflated rows: {n_base} → {len(merged)}"
)
print(f"After EDA-4 join: {len(merged)} rows")

# --- Join EDA-5: temporal_stability per (signal, position) ---
# homogeneity from EDA-5 is the temporal_stability characterization.
eda5 = pooling_decisions_df[["signal", "position", "homogeneity"]].rename(
    columns={"homogeneity": "temporal_stability"}
).copy()
merged = merged.merge(eda5, on=SIGNAL_POSITION_KEYS, how="left")
assert len(merged) == n_base, (
    f"EDA-5 join inflated rows: {n_base} → {len(merged)}"
)
print(f"After EDA-5 join: {len(merged)} rows")

# --- Derive construct_relationship_present from EDA-6 construct map ---
# A signal has a construct relationship if it appears as signal_a or signal_b
# for that position.
construct_signals_a = construct_map_df[["signal_a", "position"]].rename(
    columns={"signal_a": "signal"}
)
construct_signals_b = construct_map_df[["signal_b", "position"]].rename(
    columns={"signal_b": "signal"}
)
construct_signal_positions = (
    pd.concat([construct_signals_a, construct_signals_b], ignore_index=True)
    .drop_duplicates()
)
construct_signal_positions["construct_relationship_present"] = True

merged = merged.merge(
    construct_signal_positions, on=SIGNAL_POSITION_KEYS, how="left"
)
assert len(merged) == n_base, (
    f"construct_relationship_present join inflated rows: {n_base} → {len(merged)}"
)
merged["construct_relationship_present"] = (
    merged["construct_relationship_present"].fillna(False).astype(bool)
)
print(f"After construct join: {len(merged)} rows")

# --- Assign promotion_class via governed function ---
# signal_layer, downstream_status, temporal_stability, association_class are
# required by assign_promotion_class. Blocked rows receive NaN.
promotion_classes = []
for row in merged.to_dict(orient="records"):
    if row.get("downstream_status") == "blocked":
        promotion_classes.append(float("nan"))
    else:
        promotion_classes.append(assign_promotion_class(row))
merged["promotion_class"] = promotion_classes

print(f"Promotion class assigned. Counts:")
print(merged["promotion_class"].value_counts(dropna=False).to_string())

Base frame rows: 116
After EDA-4 join: 116 rows
After EDA-5 join: 116 rows
After construct join: 116 rows
Promotion class assigned. Counts:
promotion_class
review_signal       47
NaN                 24
context_control     18
market_context      16
core_signal          9
exposure_control     2


/var/folders/66/hjdps8cj0sd_nt4v35d18wzc0000gn/T/ipykernel_39025/3242542957.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["construct_relationship_present"].fillna(False).astype(bool)


## 4. Validate synthesis output

In [6]:
# Assert no duplicate final synthesis rows.
final_dups = int(merged[SYNTHESIS_GRAIN_KEYS].duplicated().sum())
assert final_dups == 0, (
    f"Synthesis frame has {final_dups} duplicate (signal, position, population_scope) rows"
)

# Assert no null signal names or positions.
assert merged["signal"].isna().sum() == 0, "Synthesis frame has null signal names"
assert merged["position"].isna().sum() == 0, "Synthesis frame has null positions"

# Assert promotion_class vocabulary membership for non-blocked rows.
non_blocked = merged[merged["downstream_status"] != "blocked"]
null_promo = non_blocked["promotion_class"].isna().sum()
assert null_promo == 0, (
    f"{null_promo} non-blocked rows have null promotion_class"
)
invalid_promo = set(non_blocked["promotion_class"].dropna().unique()) - PROMOTION_CLASS_VALUES
assert not invalid_promo, (
    f"Synthesis frame contains unrecognized promotion_class values: {invalid_promo}\n"
    f"Governed vocabulary: {sorted(PROMOTION_CLASS_VALUES)}"
)

# Assert no unexpected row-count inflation relative to base.
assert len(merged) == n_base, (
    f"Row count changed during synthesis: {n_base} → {len(merged)}"
)

print(f"All synthesis validations passed. Final rows: {len(merged)}")

All synthesis validations passed. Final rows: 116


## 5. Write governed output

In [7]:
# Select final output columns and drop signal_layer used only for promotion_class assignment.
synthesis_df = merged[SYNTHESIS_OUTPUT_COLS].copy()

# Deterministic sort before write.
synthesis_df = synthesis_df.sort_values(
    ["signal", "position", "population_scope"]
).reset_index(drop=True)

assert len(synthesis_df) >= 10, (
    f"Synthesis output has only {len(synthesis_df)} rows — expected >= 10 fully populated rows"
)

FINDINGS_DIR.mkdir(parents=True, exist_ok=True)
synthesis_df.to_csv(SYNTHESIS_OUTPUT_PATH, index=False)
print(f"Written: {SYNTHESIS_OUTPUT_PATH}  ({len(synthesis_df)} rows)")

Written: ../findings/eda_07_signal_synthesis.csv  (116 rows)


## 6. Summary

In [8]:
print("=" * 60)
print("EDA-7 — Signal Synthesis Summary")
print("=" * 60)

print(f"\nTotal rows: {len(synthesis_df)}")

print("\npromotion_class counts:")
promo_counts = synthesis_df["promotion_class"].value_counts(dropna=False).reindex(
    sorted(PROMOTION_CLASS_VALUES), fill_value=0
)
for label, count in promo_counts.items():
    print(f"  {label:<25} {count}")

print("\nposition counts:")
for pos, count in synthesis_df["position"].value_counts().items():
    print(f"  {pos:<10} {count}")

print("\nconstruct_relationship_present counts:")
for val, count in synthesis_df["construct_relationship_present"].value_counts().items():
    print(f"  {str(val):<10} {count}")

print("\nOutput:")
print(f"  {SYNTHESIS_OUTPUT_PATH}")
print("=" * 60)

EDA-7 — Signal Synthesis Summary

Total rows: 116

promotion_class counts:
  context_control           18
  core_signal               9
  exposure_control          2
  market_context            16
  review_signal             47

position counts:
  DEF        29
  FWD        29
  GK         29
  MID        29

construct_relationship_present counts:
  False      74
  True       42

Output:
  ../findings/eda_07_signal_synthesis.csv
